# Agents
This tutorial will walk you through how to build agents and some applications


Most of this based off of documentation provided by langchain,
see https://docs.langchain.com/oss/python/langchain/tools

The code for this tutorial was intially created with Qwen 3.6 35B A3B and was then debugged and editted 

## Setup
This tutorial is setup for local models provided through either ollama or lmstudios.

You can check requirements.txt for used modules.

You can load in the the env by calling **$ pip install -r requirements.txt** in the terminal


In [1]:
from langchain_core.tools import tool          # Decorator to define custom tools
from langchain.agents import create_agent # Factory to build the agent
from langchain_core.messages import HumanMessage, SystemMessage # Wrapper for user input   
from pydantic import BaseModel, Field
from enum import Enum
from typing import List, Dict, Any, Optional

In [ ]:
MODEL_ID:str = "gemma-4-e4b-it"
EMBEDDING_MODEL_ID:str = "text-embedding-embeddinggemma-300m-qat"
#Comment out the version you are not using
"""
#LM studios version
from langchain_openai import ChatOpenAI,OpenAIEmbeddings  # OpenAI-compatible client for LM Studio
from LmRag import RAG
# LM Studio exposes an OpenAI-compatible API on http://localhost:1234/v1 by default, Check if OCR is enabled in server settings
# - model: Must match the EXACT name of the loaded model in LM Studio (check in UI)
# - base_url: Points to LM Studio's local API endpoint
# - api_key: LM Studio doesn't require a real key; "lm-studio" is a placeholder
lm_studio_model = ChatOpenAI(
    model=MODEL_ID,  # ⚠️ Replace with your loaded model's name
    base_url="http://localhost:1234/v1",             # Default LM Studio API endpoint
    api_key="not-needed",                             # Placeholder key # type: ignore
    temperature=0.0                                  # Zero temperature for reliable math
)
embedding_model: OpenAIEmbeddings = OpenAIEmbeddings(
            model=EMBEDDING_MODEL_ID,
            base_url="http://localhost:1234/v1",
            api_key="not-needed", # type: ignore
            check_embedding_ctx_length=False
        )
rag = RAG(embedding_model=embedding_model, vector_store_path="faiss_vectorstore")
"""
"""#Ollama version
from langchain_ollama import ChatOllama      
from OllamaRag import RAG
rag = RAG(embedding_model=MODEL_ID, llm_model=MODEL_ID, vector_store_path="faiss_vectorstore")
# - model: Name of the pulled Ollama model
# - temperature: 0 ensures deterministic, consistent outputs (important for math)
ollama_model = ChatOllama(
    model=MODEL_ID,      # Change to 'llama3', 'mistral', 'phi3', etc.
    temperature=0.0,       # Lower temperature = less randomness
    num_ctx=4096           # Context window size (adjust if needed)
)
"""

"#Ollama version\nfrom langchain_ollama import ChatOllama      \n# - model: Name of the pulled Ollama model\n# - temperature: 0 ensures deterministic, consistent outputs (important for math)\nollama_model = ChatOllama(\n    model=MODEL_ID,      # Change to 'llama3', 'mistral', 'phi3', etc.\n    temperature=0.0,       # Lower temperature = less randomness\n    num_ctx=4096           # Context window size (adjust if needed)\n)\n"

## Simple agent and custom tools


Tools can defined through the @tool decorator but you need to provide enough context for the agent to use it. 

1. The function and parameters needs clearly defined name, prefer snake case (hello_world) over came case (helloWorld).

2. The parameteres and return need to be annotated (strongly typed).

3. you need a doc string after the signature to describe the tool. 

### Defining Custom Tools

In [3]:
@tool
def add(a: float, b: float) -> float:
    """Add two numbers together.
    
    Args:
        a: First operand
        b: Second operand
        
    Returns:
        The sum of a and b
    """#This is a doc string and provides a descritpion of the tool to the agent
    return a + b

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together.
    
    Args:
        a: First operand
        b: Second operand
        
    Returns:
        The product of a and b
    """
    return a * b
#Alternatively, you can override name and description in the decorator
@tool("subtract", description= "Subtract the second number from the first.")
def sub(a: float, b: float) -> float:
    return a - b




### Creating The Agent

In [4]:
tools = [add, multiply, sub]
agent = create_agent(model=lm_studio_model, tools=tools)

### Running The Agent

In [5]:
query = "Calculate (20 + 5) * 3, then subtract 10."

result = agent.invoke({"messages": [HumanMessage(content=query)]})

final_answer = result["messages"][-1].content
print("LM Studio Agent Response:")
print(final_answer)

LM Studio Agent Response:
The result is 65.


# Expert Agents

In [6]:
# -----------------------------
# TOOL: INTERFACE WITH EXPERT SYSTEM
# -----------------------------
from ExpertSystem import Car, CarDoctor
@tool
def diagnose_car(symptoms: dict) -> str:
    """
    Run the car expert system diagnosis.
    
    Args:
        symptoms: Dictionary mapping symptom names to their values. Do not include symptoms that are not present. Possible symptoms include:
                battery_voltage: ["low", "normal"],
                clicking_sound" [True, False],
                engine_starts: [True, False],
                headlights_dim: [True, False],
                overheating: [True],
                brake_fluid: ["normal", "low"],
                brake_noise: ["none", "squealing", "grinding"]
                Example: {"engine_starts": False, "brake_fluid": "low", "clicking_sound": True}
    
    Returns:
        The diagnosis output from the expert system.
    """
    print('Facts identified: ', symptoms)
    engine = CarDoctor()
    engine.reset()


    try:
        # Declare the fact using the extracted symptoms
        engine.declare(Car(**symptoms))
        engine.run()
    except Exception as e:
        return f"⚠️ Error running diagnosis: {str(e)}"
    return "successful attempt"


In [7]:
agent = create_agent(
    model=lm_studio_model,
    tools=[diagnose_car],
    system_prompt=(
        "You are an expert automotive diagnostic assistant. "
        "When a user describes car symptoms, extract them into a Python dictionary "
        "and call the 'diagnose_car' tool. "
        "Only request clarification if critical symptoms (e.g., engine_starts, brake_fluid, overheating) are missing. "
        "Format the input exactly as: diagnose_car(symptoms={'key': value, ...})"
    )
)

In [8]:
# Example 1: Direct symptom description
user_query_1 = "My car won't start. There's a clicking sound when I turn the key, and the battery is drained."

print("🔍 User Query:", user_query_1)
print("🤖 Agent Response=>")
result_1 = agent.invoke({"messages": [HumanMessage(content=user_query_1)]})

print("-" * 50)

# Example 2: Edge case (missing critical facts -> fallback rule)
user_query_2 = "The brakes are making a squealing noise when I slow down."

print("🔍 User Query:", user_query_2)
print("🤖 Agent Response=>")
result_2 = agent.invoke({"messages": [HumanMessage(content=user_query_2)]})

🔍 User Query: My car won't start. There's a clicking sound when I turn the key, and the battery is drained.
🤖 Agent Response=>
Facts identified:  {'clicking_sound': True, 'engine_starts': False, 'battery_voltage': 'low'}
Diagnosis: Battery likely dead.
Diagnosis: Starter motor may be faulty.
Cannot diagnose your car's problem. Further inspection is reuqired.
--------------------------------------------------
🔍 User Query: The brakes are making a squealing noise when I slow down.
🤖 Agent Response=>
Facts identified:  {'brake_noise': 'squealing'}
Maintenance: Brake pads worn — replace soon.
Cannot diagnose your car's problem. Further inspection is reuqired.


# RAG Agents

In [ ]:
print("-"*50)
examples = open("sampled_symptoms.txt").read().split("\n")
print(f"Sample symptom example:\n{examples}\n")  
for example in examples:
    rag.ingest_text(example, vector_store_path="faiss_vectorstore")

--------------------------------------------------
Sample symptom example:
["Malaria: I've had a high temperature, vomiting, chills, and intense itching. I also have a headache and am perspiring a lot. My discomfort has also been brought on by nausea and muscle ache.", 'Arthritis: My muscles have been quite weak, and my neck has been really stiff. My joints have swollen, making it difficult for me to move without becoming stiff. Walking has been quite uncomfortable as well.', "diabetes: I have trouble breathing, especially when exercising. I'm flushed and sweating in an unexpected way. I have yeast infections a lot.", "Malaria: I have a high fever, chills, and severe itching. In addition, I have a headache and am perspiring a lot. I've been feeling awful with nausea and muscle ache.", "Malaria: I've experienced severe itching, chills, nausea, and a high fever. Besides having a headache, I'm also perspiring a lot. I've been terrible with nausea and muscle ache.", "Arthritis: I've been f

In [10]:
@tool
def retrieve_similar_symptoms(query: str) -> List[Dict[str, Any]]:
    """Tool to retrieve similar symptoms from the vector store.
    """
    try:
        results = rag.retrieve(query)
        return [{"symptom": doc.page_content, "disease": doc.metadata.get("disease", "unknown")} for doc in results]
    except Exception as e:
        print(f"Error during retrieval: {str(e)}")
        return []

### Structured Output

In [11]:
class Disease(Enum):
    DIABETES = "diabetes"
    ARTHRITIS = "arthritis"
    MALARIA = "malaria"
class Diagnosis(BaseModel):
    disease: Disease = Field(..., description="The diagnosed disease based on the symptoms.")


In [12]:
agent = create_agent(
    model=lm_studio_model,
    tools=[retrieve_similar_symptoms],
    system_prompt=(
        "You are a tutorial showing how AI can be used as a diagnostic assistant. "
        "When a user describes their symptoms, pass the symptoms through the RAG retriever to find relevant information"
        "read the retrieved diagnoses and return the diagnosis that best matches the user's symptoms."
    ),
    response_format=Diagnosis
)

In [13]:
diabetes:str="My wound is healing more slowly these days. My feet and hands are tingling and becoming numb. I feel really fragile."
print("🔍 User Query:", diabetes)
result = agent.invoke({"messages": [HumanMessage(content=diabetes)]})
print("🤖 Agent Response=>", result["messages"][-1].content)
Arthritis:str="My muscles have been feeling really weak, and my neck has been extremely tight. I have swollen joints and find it difficult to move about without becoming stiff. It has also been really uncomfortable to walk."
print("🔍 User Query:", Arthritis)
result = agent.invoke({"messages": [HumanMessage(content=Arthritis)]})
print("🤖 Agent Response=>", result["messages"][-1].content)
Malaria:str="I have a high fever, chills, and severe itching. In addition, I have a headache and have been perspiring a lot. I've also been bothered by nausea and muscular ache."
print("🔍 User Query:", Malaria)
result = agent.invoke({"messages": [HumanMessage(content=Malaria)]})
print("🤖 Agent Response=>", result["messages"][-1].content)


🔍 User Query: My wound is healing more slowly these days. My feet and hands are tingling and becoming numb. I feel really fragile.
🤖 Agent Response=> Returning structured response: disease=<Disease.ARTHRITIS: 'arthritis'>
🔍 User Query: My muscles have been feeling really weak, and my neck has been extremely tight. I have swollen joints and find it difficult to move about without becoming stiff. It has also been really uncomfortable to walk.
🤖 Agent Response=> Returning structured response: disease=<Disease.ARTHRITIS: 'arthritis'>
🔍 User Query: I have a high fever, chills, and severe itching. In addition, I have a headache and have been perspiring a lot. I've also been bothered by nausea and muscular ache.
🤖 Agent Response=> Returning structured response: disease=<Disease.MALARIA: 'malaria'>


# Other things to look at

1. MCP servers: setup 
2. Multi turn agents
3. Multi modal agents
4. Memory management